Project : Pentaho Log Intelligence

Layer   : Bronze

Notebook: 04_Bronze_Ingestion_localhost_access

Version : 1.0

Description:
Loads raw Pentaho log files from Unity Catalog Volume

into the Bronze Delta table.

Author: Ernesto Felipe Garay Cervantes


#### Recibimiento de Parametros

In [0]:
import json

dbutils.widgets.text("archivos_nuevos","")

archivos_nuevos = json.loads(dbutils.widgets.get("archivos_nuevos"))

print("====ARCHIVOS RECIBIDOS===")
for archivo in archivos_nuevos:
    print(archivo)


### Configuracion

In [0]:
#configuracion

CATALOG = "pentaho_logs"
SCHEMA = "bronze"
VOLUME_PATH = "/Volumes/pentaho_logs/bronze/volume_localhostaccess"

BRONZE_TABLE_LOCALHOST = f"{CATALOG}.{SCHEMA}.bronze_localhostaccess"


### Validación Volumen 

In [0]:
display(dbutils.fs.ls(VOLUME_PATH))

### Lectura de archivos localhost access

In [0]:
from pyspark.sql.functions import col

VOLUME_PATH ="/Volumes/pentaho_logs/bronze/volume_localhostaccess"

archivos_path = [
    f"{VOLUME_PATH}/{archivo}"
    for archivo in archivos_nuevos
]


print("==== PATH PROXIMOS A PROCESARSE ====")

for path in archivos_path:
    print(path)



df_bronze_local_host_access = (
    spark.read
       .text(archivos_path)
       .select(
           col("value").alias("descripcion"),
        col("_metadata.file_path").alias("file_path"),
        col("_metadata.file_name").alias("file_name")
       )
    )
#display(df_bronze_local_host_access.limit(10))


### Enriquecimiento DataFrame localhost Access

In [0]:

from pyspark.sql.functions import     regexp_extract,current_timestamp,col,when,split,concat_ws
 
#separador
separador = split(col("descripcion"), r"\|")


df_bronze_local_host_access = (df_bronze_local_host_access.withColumn("aplicacion",
        regexp_extract("file_name", r"^([A-Za-z_]+)", 1))
                               .withColumn("Fecha",separador.getItem(0))
                               .withColumn("Direccion_IP",separador.getItem(1))
                               .withColumn("Peticion_servidor",separador.getItem(2))
                               .withColumn("Puerto",separador.getItem(3))
                               .withColumn("URL",separador.getItem(4))
                               .withColumn("Respuesta_Servidor",separador.getItem(6))
                               .withColumn("descripcion_log",regexp_extract(col("descripcion"),r"^\[[^\]]+\]\|[^|]*\|[^|]*\|[^|]*\|[^|]*\|[^|]*\|[^|]*\|(.*)$",1))
                                .withColumn("Timestamp",current_timestamp())  
                               )

                            

### limpiar data frame 
###df = df.drop("<campo>")
##df_bronze_catalina= df_bronze_catalina.drop("value")
#display(df_bronze_local_host_access.limit(20))


#### Validaciones DATAFRAME

In [0]:
# Número de registros
print(f"Total de líneas: {df_bronze_local_host_access.count():,}")

# Estructura
df_bronze_local_host_access.printSchema()

In [0]:
from pyspark.sql.functions import col, sum, when

df_bronze_local_host_access.select(
    sum(when(col("descripcion").isNull(), 1).otherwise(0)).alias("descripcion_null"),
    sum(when(col("file_path").isNull(), 1).otherwise(0)).alias("file_path_null"),
    sum(when(col("file_name").isNull(), 1).otherwise(0)).alias("file_name_null"),
    sum(when(col("aplicacion").isNull(), 1).otherwise(0)).alias("aplicacion_null"),
 sum(when(col("Fecha").isNull(), 1).otherwise(0)).alias("Fecha_null"),
  sum(when(col("Direccion_IP").isNull(), 1).otherwise(0)).alias("Direccion_IP_null"),
  sum(when(col("Peticion_servidor").isNull(), 1).otherwise(0)).alias("Peticion_servidor_null"),
  sum(when(col("Puerto").isNull(), 1).otherwise(0)).alias("Puerto_null"),
   sum(when(col("URL").isNull(), 1).otherwise(0)).alias("URL_null"),
   sum(when(col("Respuesta_Servidor").isNull(), 1).otherwise(0)).alias("Respuesta_Servidor_null"),
sum(when(col("descripcion_log").isNull(), 1).otherwise(0)).alias("descripcion_log_null"),
).show()




### Creación tabla BRONZE_TABLE_LOCALHOST_ACCESS

In [0]:
BRONZE_TABLE_LOCALHOST = "pentaho_logs.bronze.bronze_localHostAccess"
(
    df_bronze_local_host_access.write
        .format("delta")
        .mode("append")
        .saveAsTable(BRONZE_TABLE_LOCALHOST)
)

In [0]:
#display(spark.table(BRONZE_TABLE_LOCALHOST).limit(20))